# Retreiver Demonstration

In [15]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGREvSS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json
from beir.datasets.data_loader import GenericDataLoader
import torch.nn.functional as F

In [23]:
# Loading Packages
import pandas as pd
import numpy as np
import random
import os
import torch
import faiss
import json
import re
import shutil
import random
from tqdm import tqdm
from torch import Tensor
from dotenv import load_dotenv
from huggingface_hub import login
from torch.utils.data import DataLoader
from beir.datasets.data_loader import GenericDataLoader
from transformers import AutoTokenizer, logging, AutoModel, AutoModelForCausalLM


### Loading LLM ###
# Authenticating Token
load_dotenv('/work/mbouthil/MMATH-CM-Research-Project/token.env')
token = os.getenv('HUGGINGFACE_TOKEN')
login(token)

# Loading model and Tokenizer
model_name = "meta-llama/Llama-3.1-8B-Instruct"
# tokenizer =  AutoTokenizer.from_pretrained(model_name, token=token)
# tokenizer.pad_token = tokenizer.eos_token

# model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     dtype=torch.bfloat16,
#     device_map="auto",
#     token=token

ConnectError: [Errno 101] Network is unreachable

In [ ]:
snapshot_download(
    repo_id=model_id,
    local_dir="/work/mbouthil/models/llama-3.1-8b",
    local_dir_use_symlinks=False
)

In [16]:
data_dir = "/work/mbouthil/datasets/msmarco_synp_final"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

100%|██████████| 8841887/8841887 [01:12<00:00, 122456.05it/s]


In [17]:
print(len(corpus))

8841887


In [18]:
max_id = max([int(key) for key in corpus.keys()])

In [19]:
corpus[str(max_id)]

{'text': 'The Esperanza Fire began on October 26, 2006, in Cabazon, California, approximately two hours north of San Diego. Fueled by strong Santa Ana winds, the fire ultimately consumed 41,000 acres and destroyed 34 homes. A subsequent investigation revealed that the fire was intentionally set by an arsonist, and in 2009, Raymond Lee Oyler was sentenced to death for his role in the crime.',
 'title': ''}

In [7]:
print(corpus)

{None: {'text': None, 'title': None}}


In [27]:
data_dir = "/work/mbouthil/datasets/msmarco_synq_final"
syn_corpus, syn_queries, syn_qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

100%|██████████| 8841823/8841823 [01:15<00:00, 116813.67it/s]


In [22]:
print(len(syn_queries) - len(queries))
print(len(queries))

502939
502939


In [5]:
max_id = max([int(key) for key in queries.keys()])
print(max_id)

1185869


In [18]:
print(syn_queries[str(max_id+1)])
print(queries[str(max_id)])

What were the direct consequences of the Manhattan Project's achievement?
)what was the immediate impact of the success of the manhattan project?


In [19]:
print(print(syn_queries[str(max_id+2)]))
print(queries[str(max_id-1)])

What is the primary goal of restorative justice in addressing the consequences of a criminal offense?
None
_________ justice is designed to repair the harm to victim, the community and the offender caused by the offender criminal act. question 19 options:


In [78]:
key = list(queries.keys())[4]
print("Original query:", queries[key])
print("\n")
print("New synthetic query:", syn_queries[str(max_id+2)])

Original query: elegxo meaning


New synthetic query: What is the definition and significance of the term 'elegxo'?


In [76]:
# print(len(syn_queries))
# syn_queries['10000000'] = 'test'
# print(len(syn_queries))
# LENGTH GETS LONGER 

754408
754409


In [46]:
key = list(queries.keys())[2]
print("Original query:", queries[key])
print("\n")
print("New synthetic query:", syn_queries[str(max_id+1)])

Original query: what color is amber urine


New synthetic query: What shade of yellow is typically associated with a normal or healthy urine color?


In [8]:
# Loading Training Data:
data_dir = "/work/mbouthil/datasets/msmarco_syn_pas_1"
corpus, queries, qrels = GenericDataLoader(data_folder=data_dir).load(split="train")

100%|██████████| 9311948/9311948 [01:37<00:00, 95484.90it/s] 


In [15]:
for key, value in corpus.items():
    corpus[key] = value['text']

In [82]:
print(len(queries))
print(len(syn_queries))

print(len(syn_queries)/len(queries))

502939
754408
1.499999005843651


In [16]:
for value in corpus.values():
    print(value)
    break

{'text': 'The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated.', 'title': ''}


# Data Exploration

### Query Data Exploration

In [27]:
n = 10
i = 0

print('-Query ID--------Query Text-')
for key, value in queries.items():
    print('{'+ key, ":", value, "}")
    i += 1
    if i == n:
        break

-Query ID--------Query Text-
{1185869 : )what was the immediate impact of the success of the manhattan project? }
{1185868 : _________ justice is designed to repair the harm to victim, the community and the offender caused by the offender criminal act. question 19 options: }
{597651 : what color is amber urine }
{403613 : is autoimmune hepatitis a bile acid synthesis disorder }
{1183785 : elegxo meaning }
{312651 : how much does an average person make for tutoring }
{80385 : can you use a calculator on the compass test }
{645590 : what does physical medicine do }
{645337 : what does pending mean on listing }
{186154 : feeding rice cereal how many times per day }


In [47]:
print(f"There are a total of {len(queries):,} training queries")
print(f"There are a total of {len(test_queries):,} testing queries")
print("\n")

mean_length = np.mean([len(queries[key]) for key in queries.keys()])
print(f"The queries have a mean character length of {mean_length:.2f}")
mean_length = np.mean([len(queries[key].split()) for key in queries.keys()])
print(f"The queries have a mean word count of {mean_length:.2f}")

There are a total of 502,939 training queries
There are a total of 43 testing queries


The queries have a mean character length of 33.22
The queries have a mean word count of 5.97


### Passage Data Exporation

In [28]:
n = 10
i = 0
print('-Passage ID--------Passage Text-')
for key, value in corpus.items():
    print('{' + key, ":", value['text'], "}")
    i += 1
    if i == n:
        break

-Passage ID--------Passage Text-
{0 : The presence of communication amid scientific minds was equally important to the success of the Manhattan Project as scientific intellect was. The only cloud hanging over the impressive achievement of the atomic researchers and engineers is what their success truly meant; hundreds of thousands of innocent lives obliterated. }
{1 : The Manhattan Project and its atomic bomb helped bring an end to World War II. Its legacy of peaceful uses of atomic energy continues to have an impact on history and science. }
{2 : Essay on The Manhattan Project - The Manhattan Project The Manhattan Project was to see if making an atomic bomb possible. The success of this project would forever change the world forever making it known that something this powerful can be manmade. }
{3 : The Manhattan Project was the name for a project conducted during World War II, to develop the first atomic bomb. It refers specifically to the period of the project from 194 â¦ 2-1946 un

In [48]:
print(f"There are a total of {len(corpus):,} training passages")
print(f"There are a total of {len(test_corpus):,} testing passages")
print("\n")

mean_length = np.mean([len(corpus[key]['text']) for key in corpus.keys()])
print("The corpus (pasages) has a mean character length of {mean_length:.2f}")
mean_length = np.mean([len(corpus[key]['text'].split()) for key in corpus.keys()])
print("The corpus (pasages) has a mean word count of {mean_length:.2f}")

There are a total of 8,841,823 training passages
There are a total of 8,841,823 testing passages


The corpus (pasages) has a mean character length of {mean_length:.2f}
The corpus (pasages) has a mean word count of {mean_length:.2f}


### Qrels Data Exploration

In [32]:
n = 10
i = 0

print('-Query ID--------dict(passage ID: Score)-')
for key, value in qrels.items():
    print('{'+ key, ":", value, "}")
    i += 1
    if i == n:
        break

-Query ID--------dict(passage ID: Score)-
{1185869 : {'0': 1} }
{1185868 : {'16': 1} }
{597651 : {'49': 1} }
{403613 : {'60': 1} }
{1183785 : {'389': 1} }
{312651 : {'616': 1} }
{80385 : {'723': 1} }
{645590 : {'944': 1} }
{645337 : {'1054': 1} }
{186154 : {'1160': 1} }


In [42]:
print(len(qrels))
print(len(qrels) ==  len(queries))
print("")
print("The qrels provide the mapping of queries to relevant passage(s)")

502939
True

The qrels provide the mapping of queries to relevant passage(s)


In [43]:
# n = 200
# i = 0
# print("")
# for key, value in qrels.items():
#     if len(value) > 1:
#         print('{', key, ":", value, "}")
#     i += 1
#     if i == n:
#         break

# Retreiver Demonstration

### We begin by loading the Vector Database:

**Recall:** this vector database is constructed using the trained Passage Encoder

In [ ]:
# Loading Index
index = faiss.read_index("/work/mbouthil/MMATH-CM-Research-Project/RAG/retrieval_data/passage_v01.index")
print(f"Index contains {index.ntotal} vectors")

### Load the trained Query Encoder

In [ ]:
# Loading Query Encoder
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/MMATH-CM-Research-Project/RAG/model_weights/query_encoder_v01"
) # .to("cuda")

query_encoder.eval()

def encode_query(query:str, batch_size:int=32) -> Tensor:

    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            query, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=32
        ) #.to("cuda")

    emb = query_encoder(**inputs).last_hidden_state[:, 0]
    emb = F.normalize(emb, p=2, dim=-1)

    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 209.54it/s, Materializing param=pooler.dense.weight]                               


### Embedding the Quesetion

In [ ]:
question = queries['8']
print(question)

# Embedding Query
q_emb = encode_query([question]).detach().cpu().numpy()
# print('Embedding length: ', q_emb.shape[1])

 In humans, the normal set point for body temperature is 
Embedding length:  768


In [ ]:
K = 10
scores, ids = index.search(q_emb, K)
scores = scores[0]
ids = ids[0]

In [ ]:
for i, id in enumerate(ids):
    print(scores[i])
    print(corpus[str(id)]['text'])
    print("\n\n")

0.9989547
Thermoneutral zone. This is the range of environmental temperatures in which body temperature can be maintained by adjustment of skin blood flow alone. Normally it is kept 25-31C (77-88F) Above this zone = sweating and max vasodilation(âmetabolic activity), we have a narrow survival region in this range.



0.99892867
A personâs basal body temperature is the temperature at which the body is at immediately upon waking up from a nightâs rest. If you want to be extremely precise when measuring your body temperature, the moment you open your eyes, reach out and grab your thermometer â but donât move the rest of your body!



0.99891543
A: It is normal for body temperature to fluctuate by a degree or two Fahrenheit over the course of a day. If you remained at 98.6 all the time, you'd probably be of interest to medical researchers. You are nowhere near hypothermic. If you've h...



0.9989148
Normal human body temperature varies slightly from person to person and by the t

### It Works! 